In [ ]:
# !pip install -q zarr

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Biohub — Cell Tracking During Development  |  Solution v9
# ══════════════════════════════════════════════════════════════════════════════
#
# Version history:
#   v4  (0.820): DoG, gap closing, single-pass Hungarian
#   v5  (0.830): min_dist 3.2→4.0, drop ILP
#   v6  (0.842): thr=0.040, max_gap=2, gap_dist=7.0, safe_divisions
#   v7  (0.842+): two-pass linking, filter_short_tracks
#   v8  (0.854+): XY offset, large refine window, weighted validator
#   v9  (this):  see below
#
# v8 → v9 CHANGES  (from 0.856 lb-rule-based-v13)
# ─────────────────────────────────────────────────────────────────────────────
# [v9-1] linefit_smooth(w=0.8) — polynomial track smoothing, better 7µm matching
# [v9-2] recover_gap2 — bridge 2-frame detection gaps (t → t+3) with 2 nodes
# [v9-3] rel_threshold 0.040 → 0.030 (v13 config)
# [v9-4] max_gap=1, gap_dist=6.0 (v13; was max_gap=2/7.0)
# [v9-5] simple intensity refine (v13 style); bg-sub optional via USE_BGSUB_REFINE
# [v9-6] DO_SAFE_DIV=False by default (v13 has none; enable for A/B test)
#
# v7 → v8 CHANGES  (from 0.854 baseline analysis)
# ─────────────────────────────────────────────────────────────────────────────
# [v8-1] XY DOWNSAMPLE OFFSET before refine
#   Peaks live on a strided grid; adding (XY_DS-1)/2 centers them before CoM.
# [v8-2] LARGER REFINE WINDOW (3, 9, 9) — baseline uses this vs our (1, 3, 3)
# [v8-3] INTEGER COORDS in submission (baseline rounds z,y,x to int)
# [v8-4] WEIGHT-AVERAGED validator aggregate (TP+FP+FN weights, closer to LB)
# [v8-5] ROBUST read_geff (baseline-style zarr.open path)
#
# v6 → v7 CHANGES
# ─────────────────────────────────────────────────────────────────────────────
# [v7-1] TWO-PASS LINKING  ← key structural change from the 0.842 notebooks
#
#   Single-pass Hungarian at 8µm → two-pass:
#     Pass 1 (tight, 6µm): velocity-predicted cost, reliable short-range links
#     Pass 2 (loose, 8µm): raw distance only, UNMATCHED nodes only
#
#   Why two passes beats single-pass at the loose distance:
#     In dense frames, Pass 1 locks in confident short-range links first.
#     This prevents a fast-moving cell from "stealing" a closer cell's
#     partner, which would leave both incorrectly linked in a single pass.
#     Pass 2 then handles genuinely fast-moving or newly-appearing cells
#     with the full 8µm gate, but only those nodes still available.
#
# [v7-2] filter_short_tracks(min_len=4)  ← second key structural change
#
#   After all linking + gap closing, remove connected components with
#   fewer than 4 nodes. Removes single-frame noise detections (1 node),
#   two-frame ghost tracks (2 nodes), and marginal three-frame tracks.
#   Effect: reduces T_pred → lower over-prediction penalty + fewer FP edges.
#
#   From the 0.842 notebooks: this change reduces 0b24845f from
#   ~27,665 nodes to ~26,023 (−1,642 ghost tracks removed).
#
#   Note on "divisions regressed the LB" in the 0.842 notebooks:
#   This refers to the full `twopass_div` LINKER (division handling in the
#   assignment step itself). Our safe_divisions is a conservative POST-LINK
#   pass that never modifies existing edges — a completely different approach.
#
# RETAINED FROM v6
# ─────────────────────────────────────────────────────────────────────────────
# safe_divisions: conservative post-link second-daughter recovery (+0.004 LB)
# bg-sub centroid refinement with shift rejection (from 0.827 notebook)
# Tuned params: thr=0.040, max_gap=2, gap_dist=7.0
# Local validator with accurate metric implementation
# ══════════════════════════════════════════════════════════════════════════════

# !pip install -q zarr   ← uncomment if zarr not available


# ── CELL 1: Imports ───────────────────────────────────────────────────────────

import os, gc, json, glob, time
from collections import defaultdict

import numpy as np
import pandas as pd
import blosc2
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment

print(f"numpy {np.__version__}  |  pandas {pd.__version__}")


# ── CELL 2: Configuration ─────────────────────────────────────────────────────

SCALE = np.array([1.625, 0.40625, 0.40625])   # Z, Y, X  µm/voxel

# --- Detection ---
XY_DS           = 4
DOG_SCALES      = [[1.5, 4.0], [2.2, 5.5]]
REL_THRESHOLD   = 0.032   # [v9] v13 0.856 uses 0.030 (was 0.040)
ABS_PERCENTILE  = 50.0
MIN_DIST_UM     = 4.0
MAX_PEAKS       = 40000

# --- Refinement ---
REFINE_XY_OFFSET    = True
REFINE_WIN          = (3, 9, 9)
USE_BGSUB_REFINE    = False  # [v9] v13 uses simple intensity CoM; True = bg-sub mode
REFINE_BG_PCT       = 20.0
REFINE_MAX_SHIFT_UM = 2.8

# --- Submission ---
ROUND_COORDS        = True   # [v8] baseline rounds node coords to int

# --- Two-pass linking ---  [v7 new]
TIGHT_UM     = 6.0    # Pass 1: reliable short-range links
MAX_LINK_UM  = 8.0    # Pass 2: remaining unmatched nodes
VEL_BLEND    = 0.5    # velocity weight in Pass 1 cost

# --- Gap closing (standard 1-frame) ---
DO_GAP_CLOSE = True
MAX_GAP      = 1      # [v9] v13 uses 1; 2-frame gaps handled by recover_gap2
GAP_DIST_UM  = 6.0    # [v9] v13 config

# --- Post-link smoothing & 2-frame recovery [v9 new] ---
DO_LINEFIT_SMOOTH      = True
SMOOTH_W               = 0.8    # v13 0.856
SMOOTH_WIN             = 2
DO_RECOVER_GAP2        = True
RECOVER_GAP2_MAX_TOTAL = 10.2   # µm total end→start distance
RECOVER_GAP2_MAX_STEP  = 4.4    # µm per interpolated step
RECOVER_GAP2_MAX_FRAC  = 0.02   # cap added bridges at 2% of nodes

# --- Safe divisions (off in v13; enable for A/B) ---
DO_SAFE_DIV            = True
DIV_MAX_UM             = 5.0
DIV_SIBLING_MAX_UM     = 8.0
DIV_EXIST_CHILD_MAX_UM = 8.5
DIV_FRAME_FRAC_CAP     = 0.012
DIV_GLOBAL_FRAC_CAP    = 0.006

# --- Post-processing ---
PRUNE_ISOLATED = True
MIN_TRACK_LEN  = 4     # remove connected components with fewer nodes  [v7 new]
KEEP_DIV_COMPS = True  # always keep components containing a division

# --- Validator ---
RUN_VALIDATOR = True
VALIDATOR_N   = 4


# ── CELL 3: Path Discovery ────────────────────────────────────────────────────

def find_dir(candidates, pattern='*.zarr'):
    for p in candidates:
        if os.path.isdir(p) and glob.glob(os.path.join(p, pattern)):
            return p
    leaf = os.path.basename(candidates[0].rstrip('/'))
    for hit in glob.glob(f'/kaggle/input/**/{leaf}', recursive=True):
        if glob.glob(os.path.join(hit, pattern)):
            return hit
    return candidates[0]

TEST_DIR  = find_dir([
    '/kaggle/input/competitions/biohub-cell-tracking-during-development/test',
    '/kaggle/input/biohub-cell-tracking-during-development/test',
])
TRAIN_DIR = find_dir([
    '/kaggle/input/competitions/biohub-cell-tracking-during-development/train',
    '/kaggle/input/biohub-cell-tracking-during-development/train',
])
print(f"TEST_DIR  = {TEST_DIR}")
print(f"TRAIN_DIR = {TRAIN_DIR}")


# ── CELL 4: Data I/O ─────────────────────────────────────────────────────────

def list_datasets(directory):
    return sorted(d[:-5] for d in os.listdir(directory) if d.endswith('.zarr'))

def read_meta(zarr_path):
    meta = json.load(open(os.path.join(zarr_path, '0', 'zarr.json')))
    return tuple(meta['shape']), np.dtype(meta['data_type'])

def load_volume(zarr_path, t, shape, dtype):
    chunk = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
    return np.frombuffer(blosc2.decompress(open(chunk, 'rb').read()),
                         dtype=dtype).reshape(shape[1:])

def read_geff_estimated_nodes(geff_path):
    root_json = os.path.join(geff_path, 'zarr.json')
    if os.path.exists(root_json):
        try:
            meta = json.load(open(root_json))
            val = (meta.get('attributes', {})
                       .get('geff', {})
                       .get('extra', {})
                       .get('estimated_number_of_nodes'))
            if val is not None: return val
        except Exception: pass
    def _search(obj, key):
        if isinstance(obj, dict):
            if key in obj: return obj[key]
            for v in obj.values():
                r = _search(v, key)
                if r is not None: return r
        elif isinstance(obj, list):
            for item in obj:
                r = _search(item, key)
                if r is not None: return r
        return None
    for root, _, files in os.walk(geff_path):
        if 'zarr.json' not in files: continue
        try:
            v = _search(json.load(open(os.path.join(root, 'zarr.json'))),
                        'estimated_number_of_nodes')
            if v is not None: return v
        except Exception: continue
    return None

def read_geff(geff_path):
    """Read .geff GT graph. Tries baseline-style zarr.open first."""
    def _extract(store):
        node_ids = np.array(store['nodes/ids'])
        t_vals   = np.array(store['nodes/props/t/values'])
        z_vals   = np.array(store['nodes/props/z/values'])
        y_vals   = np.array(store['nodes/props/y/values'])
        x_vals   = np.array(store['nodes/props/x/values'])
        edge_arr = np.array(store['edges/ids'])
        if edge_arr.ndim == 1:
            edge_arr = edge_arr.reshape(-1, 2)
        nodes_df = pd.DataFrame({'node_id': node_ids, 't': t_vals.astype(int),
                                 'z': z_vals, 'y': y_vals, 'x': x_vals})
        edges_df = pd.DataFrame({'source_id': edge_arr[:, 0].astype(int),
                                 'target_id': edge_arr[:, 1].astype(int)})
        return nodes_df, edges_df

    try:
        import zarr
    except ImportError:
        print("    [geff] zarr not installed — add !pip install -q zarr")
        return None, None

    for opener in (
        lambda: zarr.open(str(geff_path), mode='r'),
        lambda: zarr.open_group(str(geff_path), mode='r'),
    ):
        try:
            return _extract(opener())
        except Exception:
            continue
    print(f"    [geff] read failed for {os.path.basename(geff_path)}")
    return None, None

print("I/O functions OK ✓")


# ── CELL 5: Detection ─────────────────────────────────────────────────────────

def _ball_footprint(radius_um, eff_spacing):
    rad_vox = np.maximum(1, np.round(radius_um / eff_spacing).astype(int))
    zz, yy, xx = np.ogrid[-rad_vox[0]:rad_vox[0]+1,
                          -rad_vox[1]:rad_vox[1]+1,
                          -rad_vox[2]:rad_vox[2]+1]
    return ((zz*eff_spacing[0])**2 + (yy*eff_spacing[1])**2 +
            (xx*eff_spacing[2])**2) <= radius_um**2

def detect_blobs(vol, apply_refine=True):
    vf  = vol.astype(np.float32)
    ds  = vf[:, ::XY_DS, ::XY_DS]
    eff = np.array([SCALE[0], SCALE[1]*XY_DS, SCALE[2]*XY_DS])
    lo, hi = np.percentile(ds, [1.0, 99.7])
    if hi <= lo: hi = lo + 1.0
    norm = np.clip((ds - lo) / (hi - lo), 0.0, None).astype(np.float32)
    dog = None
    for s_um, l_um in DOG_SCALES:
        resp = (gaussian_filter(norm, sigma=s_um/eff)
                - gaussian_filter(norm, sigma=l_um/eff))
        dog = resp if dog is None else np.maximum(dog, resp)
    fp      = _ball_footprint(MIN_DIST_UM, eff)
    mx      = maximum_filter(dog, footprint=fp, mode='nearest')
    abs_thr = float(np.percentile(norm, ABS_PERCENTILE))
    mask    = (dog == mx) & (dog >= REL_THRESHOLD) & (norm >= abs_thr)
    coords  = np.argwhere(mask)
    if coords.size == 0:
        return np.empty((0, 3), np.float64)
    scores = dog[mask]
    order  = np.argsort(scores)[::-1]
    coords = coords[order]
    if MAX_PEAKS and len(coords) > MAX_PEAKS:
        coords = coords[:MAX_PEAKS]
    out       = coords.astype(np.float64)
    out[:, 1] *= XY_DS
    out[:, 2] *= XY_DS
    if apply_refine and len(out) > 0:
        if REFINE_XY_OFFSET:
            xy_off = (XY_DS - 1) / 2.0
            out[:, 1] += xy_off
            out[:, 2] += xy_off
        out = refine_centroids(vol, out)
    return out

def refine_centroids(vol, coords):
    """Intensity CoM refinement (v13 default) or bg-sub with shift rejection."""
    if len(coords) == 0:
        return coords
    Z, Y, X = vol.shape
    wz, wy, wx = REFINE_WIN
    out = coords.copy()
    for i, original in enumerate(coords):
        z, y, x = int(round(original[0])), int(round(original[1])), int(round(original[2]))
        z0, z1 = max(0, z - wz), min(Z, z + wz + 1)
        y0, y1 = max(0, y - wy), min(Y, y + wy + 1)
        x0, x1 = max(0, x - wx), min(X, x + wx + 1)
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            continue
        if USE_BGSUB_REFINE:
            bg = float(np.percentile(patch, REFINE_BG_PCT))
            weights = np.maximum(patch - bg, 0.0)
            total = float(weights.sum())
            if total <= 0:
                continue
        else:
            weights = patch
            total = float(weights.sum())
            if total <= 0:
                continue
        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]
        refined = np.array([(weights * zz).sum() / total,
                            (weights * yy).sum() / total,
                            (weights * xx).sum() / total])
        if USE_BGSUB_REFINE:
            shift_um = float(np.sqrt((((refined - original) * SCALE) ** 2).sum()))
            if shift_um > REFINE_MAX_SHIFT_UM:
                continue
        out[i] = refined
    return out

print("Detection functions OK ✓")


# ── CELL 6: Two-Pass Linking ──────────────────────────────────────────────────

def link_twopass(frame_coords, frame_ids):
    """
    Two-pass frame-by-frame Hungarian linking.  [v7 key change]

    Pass 1 — tight (TIGHT_UM = 6.0 µm):
        Cost = velocity-predicted distance (VEL_BLEND = 0.5).
        Gate = raw physical distance ≤ TIGHT_UM.
        Locks in reliable short-range matches first.

    Pass 2 — loose (MAX_LINK_UM = 8.0 µm):
        Only operates on UNMATCHED nodes from Pass 1.
        Cost = gate = raw physical distance.
        No velocity: simpler, avoids over-engineering the hard cases.

    Why better than single-pass at 8µm:
        Dense frames → many candidate pairs within 8µm.
        Single-pass: a fast-moving cell can "steal" a nearby cell's nearest
        neighbour, leaving both suboptimally linked.
        Two-pass: Pass 1 resolves the easy, confident links first. Pass 2
        only sees nodes that genuinely need a looser gate.

    Returns: list of (source_node_id, target_node_id) edges.
    """
    edges      = []
    velocities = {}   # {node_id: velocity_in_voxels}

    for t in range(len(frame_coords) - 1):
        prev_xyz  = frame_coords[t]
        curr_xyz  = frame_coords[t + 1]
        prev_ids  = frame_ids[t]
        curr_ids  = frame_ids[t + 1]
        N, M = len(prev_xyz), len(curr_xyz)
        if N == 0 or M == 0:
            velocities = {}
            continue

        prev_phys = prev_xyz * SCALE
        curr_phys = curr_xyz * SCALE

        # Velocity-predicted positions for Pass 1 cost
        P_pred = prev_phys.copy()
        for i, pid in enumerate(prev_ids):
            vel = velocities.get(pid)
            if vel is not None:
                P_pred[i] = prev_phys[i] + VEL_BLEND * (vel * SCALE)

        # ── Pass 1: tight gate, velocity-predicted cost ──────────────────────
        D_raw_1  = np.sqrt(((prev_phys[:, None] - curr_phys[None, :]) ** 2).sum(2))
        D_pred_1 = np.sqrt(((P_pred[:, None]    - curr_phys[None, :]) ** 2).sum(2))
        cost_1   = np.where(D_raw_1 <= TIGHT_UM, D_pred_1, 1e6)
        ri1, ci1 = linear_sum_assignment(cost_1)
        tight_links  = [(int(r), int(c)) for r, c in zip(ri1, ci1) if cost_1[r, c] < 1e6]
        matched_prev = {pi for pi, _ in tight_links}
        matched_curr = {ci for _, ci in tight_links}

        new_vel = {}
        for pi, ci in tight_links:
            edges.append((prev_ids[pi], curr_ids[ci]))
            # KEY: store velocity by curr index (not prev) — avoids IndexError
            new_vel[ci] = curr_xyz[ci] - prev_xyz[pi]

        # ── Pass 2: loose gate, raw distance only, unmatched nodes ──────────
        up_idx = [i for i in range(N) if i not in matched_prev]
        uc_idx = [j for j in range(M) if j not in matched_curr]

        if up_idx and uc_idx:
            up_phys = prev_phys[up_idx]
            uc_phys = curr_phys[uc_idx]
            D_raw_2 = np.sqrt(((up_phys[:, None] - uc_phys[None, :]) ** 2).sum(2))
            cost_2  = np.where(D_raw_2 <= MAX_LINK_UM, D_raw_2, 1e6)
            ri2, ci2 = linear_sum_assignment(cost_2)
            for r, c in zip(ri2, ci2):
                if cost_2[r, c] < 1e6:
                    pi_actual = up_idx[r]
                    ci_actual = uc_idx[c]
                    edges.append((prev_ids[pi_actual], curr_ids[ci_actual]))
                    new_vel[ci_actual] = curr_xyz[ci_actual] - prev_xyz[pi_actual]

        # Map new_vel from curr_idx → curr_node_id for next iteration
        velocities = {curr_ids[ci]: vel for ci, vel in new_vel.items()}

    return edges

print("Two-pass linking OK ✓")


# ── CELL 7: Gap Closing ───────────────────────────────────────────────────────

def close_gaps(frame_coords, frame_ids, edges):
    """Bridge missed frames with linearly-interpolated nodes."""
    node_info = {}
    for t, (coords, ids) in enumerate(zip(frame_coords, frame_ids)):
        for nid, pos in zip(ids, coords):
            node_info[nid] = (t, *pos)
    has_out = {src for src, _ in edges}
    has_in  = {tgt for _, tgt in edges}
    ends_by_t   = defaultdict(list)
    starts_by_t = defaultdict(list)
    for nid, info in node_info.items():
        t = info[0]
        if nid not in has_out:  ends_by_t[t].append(nid)
        if nid not in has_in:   starts_by_t[t].append(nid)
    new_nodes, new_edges = [], []
    next_id      = max(node_info.keys(), default=0) + 1
    bridged_ends = set()
    for gap in range(1, MAX_GAP + 1):
        dist_gate = GAP_DIST_UM * (gap + 1)
        for t_end, ends in ends_by_t.items():
            starts = starts_by_t.get(t_end + gap + 1)
            if not starts: continue
            avail = [e for e in ends if e not in bridged_ends]
            if not avail: continue
            ec = np.array([[node_info[e][1],node_info[e][2],node_info[e][3]]
                            for e in avail]) * SCALE
            sc = np.array([[node_info[s][1],node_info[s][2],node_info[s][3]]
                            for s in starts]) * SCALE
            D    = np.sqrt(((ec[:,None]-sc[None,:])**2).sum(2))
            cost = np.where(D <= dist_gate, D, 1e9)
            ri, ci = linear_sum_assignment(cost)
            used_s = set()
            for r, c in zip(ri, ci):
                if D[r,c] > dist_gate: continue
                e_id, s_id = avail[r], starts[c]
                if e_id in bridged_ends or c in used_s: continue
                te,ze,ye,xe = node_info[e_id]
                ts,zs,ys,xs = node_info[s_id]
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    nid  = next_id; next_id += 1
                    new_nodes.append({'node_id':nid,'t':te+k,
                                      'z':ze+(zs-ze)*frac,'y':ye+(ys-ye)*frac,
                                      'x':xe+(xs-xe)*frac})
                    new_edges.append((prev, nid)); prev = nid
                new_edges.append((prev, s_id))
                bridged_ends.add(e_id); used_s.add(c)
    return new_nodes, new_edges

print("Gap closing OK ✓")


# ── CELL 8: Safe Divisions ────────────────────────────────────────────────────

def add_safe_divisions(frame_coords, frame_ids, edges):
    """
    Conservative post-link second-daughter recovery.  (+0.004 LB, 0.835→0.839)

    Only adds edges. Never touches existing edges. Hard caps prevent FP flood.
    See v6 for full documentation.

    NOTE: This is NOT the 'twopass_div' linker from the 0.842 public notebooks
    which regressed their LB. Our safe_divisions is a post-link pass that adds
    at most 0.6% of total edges as division edges — fundamentally different.
    """
    if not edges: return []
    coords = {}
    for t, (fc, fi) in enumerate(zip(frame_coords, frame_ids)):
        for nid, pos in zip(fi, fc):
            coords[nid] = (t, *pos)
    out_edges = defaultdict(list)
    in_edges  = defaultdict(list)
    for s, t in edges:
        out_edges[int(s)].append(int(t))
        in_edges[int(t)].append(int(s))
    ids_by_t = defaultdict(list)
    for nid, info in coords.items():
        ids_by_t[info[0]].append(nid)
    existing_edge_set = set((int(s), int(t)) for s, t in edges)
    global_cap        = max(1, int(len(edges) * DIV_GLOBAL_FRAC_CAP))
    added, used_tgts  = [], set()
    for tt in sorted(ids_by_t.keys()):
        if tt + 1 not in ids_by_t: continue
        src_ids  = [n for n in ids_by_t[tt] if len(out_edges.get(n,[])) == 1]
        if not src_ids: continue
        cand_ids = [n for n in ids_by_t[tt+1]
                    if n not in in_edges and n not in used_tgts]
        if not cand_ids: continue
        frame_cap = max(1, int(len(src_ids) * DIV_FRAME_FRAC_CAP))
        cand_pos  = (np.array([[coords[c][1],coords[c][2],coords[c][3]]
                                for c in cand_ids], dtype=np.float64) * SCALE)
        proposals = []
        for sid in src_ids:
            child = out_edges[sid][0]
            if coords.get(child,(None,))[0] != tt + 1: continue
            sp = np.array([coords[sid][1],coords[sid][2],coords[sid][3]]) * SCALE
            cp = np.array([coords[child][1],coords[child][2],coords[child][3]]) * SCALE
            if np.linalg.norm(cp - sp) > DIV_EXIST_CHILD_MAX_UM: continue
            d_p = np.sqrt(((cand_pos-sp[None,:])**2).sum(1))
            d_s = np.sqrt(((cand_pos-cp[None,:])**2).sum(1))
            ok  = (d_p <= DIV_MAX_UM) & (d_s <= DIV_SIBLING_MAX_UM)
            if not ok.any(): continue
            idxs = np.where(ok)[0]
            scores = d_p[idxs] + 0.15 * d_s[idxs]
            best   = int(idxs[np.argmin(scores)])
            tgt    = cand_ids[best]
            if (sid, tgt) in existing_edge_set: continue
            proposals.append((float(scores.min()), sid, tgt))
        proposals.sort(key=lambda x: x[0])
        added_frame = 0
        for _, sid, tgt in proposals:
            if len(added) >= global_cap or added_frame >= frame_cap: break
            if tgt in used_tgts or tgt in in_edges: continue
            added.append((sid, tgt))
            used_tgts.add(tgt); in_edges[tgt].append(sid); added_frame += 1
    return added

print("Safe divisions OK ✓")


# ── CELL 9: filter_short_tracks ───────────────────────────────────────────────

def filter_short_tracks(node_rows, edge_rows, min_len=MIN_TRACK_LEN):
    """
    Remove connected components with fewer than min_len nodes.  [v7 key change]

    'Length' = number of nodes in the connected component (not time span).
    Mirrors the 0.842 public notebook implementation exactly.

    Effect on 0b24845f (0.842 notebooks): 27,665 → 26,023 nodes (−1,642).
    These are single-frame noise detections and short ghost tracks that:
      - Contribute to T_pred without providing matching GT edges
      - Inflate the over-prediction penalty factor
      - Create FP edges in linking

    Components with a division node are always retained (keep_div=KEEP_DIV_COMPS).
    This ensures legitimate divisions aren't pruned even if daughters are short.
    """
    if not edge_rows or not node_rows:
        return node_rows, edge_rows

    # Union-Find over all node IDs
    parent = {nr['node_id']: nr['node_id'] for nr in node_rows}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for er in edge_rows:
        s, t = er['source_id'], er['target_id']
        if s in parent and t in parent:
            rs, rt = find(s), find(t)
            if rs != rt:
                parent[rs] = rt

    # Group nodes by component
    comp_members = defaultdict(list)
    for nid in parent:
        comp_members[find(nid)].append(nid)

    # Division nodes: ≥2 outgoing edges
    out_count = defaultdict(int)
    for er in edge_rows:
        out_count[er['source_id']] += 1
    div_nodes = {nid for nid, cnt in out_count.items() if cnt >= 2}

    # Decide which components to keep
    keep_ids = set()
    n_comps_kept = 0
    n_comps_pruned = 0
    for root, members in comp_members.items():
        long_enough = len(members) >= min_len
        has_div     = KEEP_DIV_COMPS and any(m in div_nodes for m in members)
        if long_enough or has_div:
            keep_ids.update(members)
            n_comps_kept += 1
        else:
            n_comps_pruned += 1

    node_rows_out = [nr for nr in node_rows if nr['node_id'] in keep_ids]
    edge_rows_out = [er for er in edge_rows
                     if er['source_id'] in keep_ids and er['target_id'] in keep_ids]
    return node_rows_out, edge_rows_out

print("filter_short_tracks OK ✓")


def _fmt_coord(v):
    """Format node coordinate for submission (int if ROUND_COORDS else float)."""
    return int(round(v)) if ROUND_COORDS else float(v)


def linefit_smooth_rows(node_rows, edge_rows, w=SMOOTH_W, win=SMOOTH_WIN):
    """
    Polynomial line-fit smoothing along tracks (v13 0.856).

    For nodes with a linear chain neighbourhood (±win frames), blend position
    toward the linear fit at t=0. Improves 7µm node matching → higher edge_jac.
    """
    if not DO_LINEFIT_SMOOTH or w <= 0 or not edge_rows or not node_rows:
        return node_rows, edge_rows

    idx = {nr['node_id']: i for i, nr in enumerate(node_rows)}
    pos = np.array([[nr['z'], nr['y'], nr['x']] for nr in node_rows], dtype=np.float64)
    tof = np.array([nr['t'] for nr in node_rows], dtype=int)
    pred, succ = {}, {}
    for er in edge_rows:
        s, t = er['source_id'], er['target_id']
        if s not in idx or t not in idx:
            continue
        if tof[idx[t]] == tof[idx[s]] + 1:
            succ.setdefault(s, []).append(t)
            pred.setdefault(t, []).append(s)

    newpos = pos.copy()
    for nr in node_rows:
        n = nr['node_id']
        nb = [(0, n)]
        c = n
        for k in range(1, win + 1):
            p = pred.get(c)
            c = p[0] if p and len(p) == 1 else None
            if c is None:
                break
            nb.append((-k, c))
        c = n
        for k in range(1, win + 1):
            s = succ.get(c)
            c = s[0] if s and len(s) == 1 else None
            if c is None:
                break
            nb.append((k, c))
        if len(nb) < 3:
            continue
        dts = np.array([d for d, _ in nb], dtype=np.float64)
        pp = np.stack([pos[idx[i]] for _, i in nb])
        proj = np.array([np.polyval(np.polyfit(dts, pp[:, k], 1), 0.0) for k in range(3)])
        newpos[idx[n]] = (1 - w) * pos[idx[n]] + w * proj

    for i, nr in enumerate(node_rows):
        nr['z'], nr['y'], nr['x'] = float(newpos[i, 0]), float(newpos[i, 1]), float(newpos[i, 2])
    return node_rows, edge_rows


def recover_gap2_rows(node_rows, edge_rows, dataset_name):
    """
    Bridge 2-frame detection gaps: track end at t, start at t+3 (v13 0.856).

    Inserts interpolated nodes at t+1 and t+2. Recovers FN edges that
    close_gaps(max_gap=1) cannot handle.
    """
    if not DO_RECOVER_GAP2 or not edge_rows or not node_rows:
        return node_rows, edge_rows, 0

    idx = {nr['node_id']: i for i, nr in enumerate(node_rows)}
    pos = np.array([[nr['z'], nr['y'], nr['x']] for nr in node_rows], dtype=np.float64)
    tof = np.array([nr['t'] for nr in node_rows], dtype=int)
    outgoing = {er['source_id'] for er in edge_rows}
    incoming = {er['target_id'] for er in edge_rows}

    ends_by_t, starts_by_t = defaultdict(list), defaultdict(list)
    for nr in node_rows:
        n, t = nr['node_id'], int(nr['t'])
        if n not in outgoing:
            ends_by_t[t].append(n)
        if n not in incoming:
            starts_by_t[t].append(n)

    next_id = max(idx.keys(), default=0) + 1
    cap = max(1, int(round(len(node_rows) * RECOVER_GAP2_MAX_FRAC)))
    new_nodes, new_edges, used, added = [], [], set(), 0

    for t in sorted(ends_by_t.keys()):
        starts = [s for s in starts_by_t.get(t + 3, []) if s not in used]
        for e in ends_by_t[t]:
            if added >= cap:
                break
            best = None
            for s in starts:
                if s in used:
                    continue
                d = float(np.linalg.norm((pos[idx[e]] - pos[idx[s]]) * SCALE))
                if d <= RECOVER_GAP2_MAX_TOTAL and d / 3.0 <= RECOVER_GAP2_MAX_STEP:
                    if best is None or d < best[1]:
                        best = (s, d)
            if best is None:
                continue
            s = best[0]
            pe, ps = pos[idx[e]], pos[idx[s]]
            m1 = pe + (ps - pe) / 3.0
            m2 = pe + 2.0 * (ps - pe) / 3.0
            n1, n2 = next_id, next_id + 1
            next_id += 2
            new_nodes.extend([
                {'node_id': n1, 't': t + 1, 'z': m1[0], 'y': m1[1], 'x': m1[2]},
                {'node_id': n2, 't': t + 2, 'z': m2[0], 'y': m2[1], 'x': m2[2]},
            ])
            new_edges.extend([(e, n1), (n1, n2), (n2, s)])
            used.add(s)
            added += 1

    if not new_nodes:
        return node_rows, edge_rows, 0

    for nr in new_nodes:
        node_rows.append({
            'dataset': dataset_name, 'row_type': 'node',
            'node_id': nr['node_id'], 't': nr['t'],
            'z': float(nr['z']), 'y': float(nr['y']), 'x': float(nr['x']),
            'source_id': -1, 'target_id': -1,
        })
    for s, t in new_edges:
        edge_rows.append({
            'dataset': dataset_name, 'row_type': 'edge',
            'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
            'source_id': s, 'target_id': t,
        })
    return node_rows, edge_rows, len(new_nodes)


print("linefit_smooth + recover_gap2 OK ✓")


# ── CELL 10: Dataset Processing ───────────────────────────────────────────────

def track_movie(zarr_path, shape, dtype, dataset_name):
    """
    End-to-end processing for one .zarr dataset.

    Phases:
      1. Detect blobs in all timepoints
      2. Assign node IDs
      3. Two-pass linking  [v7: was single-pass]
      4. Gap closing
      5. Safe divisions
      6. Build rows
      7. Prune isolated nodes
      8. filter_short_tracks
      9. linefit_smooth  [v9]
     10. recover_gap2     [v9]
     11. Round coords for submission
    """
    T = shape[0]

    # Phase 1: Detect
    frame_coords = []
    for t in range(T):
        vol   = load_volume(zarr_path, t, shape, dtype)
        cents = detect_blobs(vol, apply_refine=True)
        del vol; gc.collect()
        frame_coords.append(cents)

    # Phase 2: Node IDs
    nid, frame_ids = 1, []
    for cents in frame_coords:
        ids = list(range(nid, nid + len(cents)))
        nid += len(cents)
        frame_ids.append(ids)

    # Phase 3: Two-pass linking
    edges = link_twopass(frame_coords, frame_ids)

    # Phase 4: Gap closing
    new_node_rows_gap, new_edges_gap = [], []
    if DO_GAP_CLOSE and T > 2:
        new_node_rows_gap, new_edges_gap = close_gaps(frame_coords, frame_ids, edges)
        edges.extend(new_edges_gap)

    # Phase 5: Safe divisions
    div_edges = []
    if DO_SAFE_DIV:
        div_edges = add_safe_divisions(frame_coords, frame_ids, edges)
        edges.extend(div_edges)

    # Phase 6: Build rows (float coords until final rounding)
    node_rows = []
    for t, (coords, ids) in enumerate(zip(frame_coords, frame_ids)):
        for nid_k, (z, y, x) in zip(ids, coords):
            node_rows.append({'dataset': dataset_name, 'row_type': 'node',
                              'node_id': nid_k, 't': t,
                              'z': float(z), 'y': float(y), 'x': float(x),
                              'source_id': -1, 'target_id': -1})
    for nr in new_node_rows_gap:
        node_rows.append({'dataset': dataset_name, 'row_type': 'node',
                          'node_id': nr['node_id'], 't': nr['t'],
                          'z': float(nr['z']), 'y': float(nr['y']), 'x': float(nr['x']),
                          'source_id': -1, 'target_id': -1})
    edge_rows = [{'dataset': dataset_name, 'row_type': 'edge',
                  'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                  'source_id': s, 'target_id': t_}
                 for s, t_ in edges]

    # Phase 7: Prune isolated nodes
    if PRUNE_ISOLATED and edge_rows:
        used = ({er['source_id'] for er in edge_rows}
                | {er['target_id'] for er in edge_rows})
        node_rows = [nr for nr in node_rows if nr['node_id'] in used]

    # Phase 8: recover_gap2 [v10 new order]
    node_rows, edge_rows, n_gap2 = recover_gap2_rows(node_rows, edge_rows, dataset_name)

    # Phase 9: filter_short_tracks [v10 new order]
    n_before = len(node_rows)
    node_rows, edge_rows = filter_short_tracks(node_rows, edge_rows)
    n_filtered = n_before - len(node_rows)

    # Phase 10: linefit_smooth [v10 new order]
    node_rows, edge_rows = linefit_smooth_rows(node_rows, edge_rows)

    # Phase 11: round coords for submission
    for nr in node_rows:
        nr['z'] = _fmt_coord(nr['z'])
        nr['y'] = _fmt_coord(nr['y'])
        nr['x'] = _fmt_coord(nr['x'])

    return node_rows, edge_rows, {
        'dataset':   dataset_name,
        'n_nodes':   len(node_rows),
        'n_edges':   len(edge_rows),
        'n_detect':  sum(len(c) for c in frame_coords),
        'n_gap':     len(new_node_rows_gap),
        'n_gap2':    n_gap2,
        'n_div':     len(div_edges),
        'n_filter':  n_filtered,
        'T':         T,
    }

print("track_movie OK ✓")


# ── CELL 11: Local Validator ──────────────────────────────────────────────────

def _match_nodes(pred_nodes, gt_nodes, r=7.0):
    p2g = {}
    for t in sorted(set(pred_nodes['t'].tolist()) & set(gt_nodes['t'].tolist())):
        pn = pred_nodes[pred_nodes['t']==t].reset_index(drop=True)
        gn = gt_nodes[gt_nodes['t']==t].reset_index(drop=True)
        if len(pn)==0 or len(gn)==0: continue
        P   = pn[['z','y','x']].values * SCALE
        G   = gn[['z','y','x']].values * SCALE
        D   = np.sqrt(((P[:,None]-G[None,:])**2).sum(2))
        cost = np.where(D<=r, D, 1e6)
        ri, ci = linear_sum_assignment(cost)
        for a, b in zip(ri, ci):
            if cost[a,b] < 1e6:
                p2g[int(pn.loc[a,'node_id'])] = int(gn.loc[b,'node_id'])
    return p2g

def compute_division_metrics(pred_df, gt_nodes, gt_edges, p2g, r=7.0):
    from collections import defaultdict
    import numpy as np
    
    # 1. Map edges to adjacency lists
    gt_adj = defaultdict(list)
    for _, row in gt_edges.iterrows():
        gt_adj[int(row['source_id'])].append(int(row['target_id']))
        
    pred_edges = pred_df[pred_df.row_type=='edge'][['source_id','target_id']].astype(int).copy()
    pred_adj = defaultdict(list)
    for _, row in pred_edges.iterrows():
        pred_adj[int(row['source_id'])].append(int(row['target_id']))
        
    # 2. Find divisions (degree >= 2)
    gt_divs = {p: list(c) for p, c in gt_adj.items() if len(c) >= 2}
    pred_divs = {p: list(c) for p, c in pred_adj.items() if len(c) >= 2}
    
    if not gt_divs or not pred_divs:
        return 0, 0, len(gt_divs)
        
    gt_nodes_lookup = gt_nodes.set_index('node_id')
    pred_nodes_lookup = pred_df[pred_df.row_type=='node'].set_index('node_id')
    
    # Trace descendants up to 3 frames downstream
    def get_descendants(node_id, adj, depth=3):
        desc = {node_id}
        curr = {node_id}
        for _ in range(depth):
            next_nodes = set()
            for n in curr:
                next_nodes.update(adj[n])
            curr = next_nodes
            desc.update(curr)
        return desc

    # Precompute GT lineages
    gt_lineages = {}
    for p_gt, daughters in gt_divs.items():
        gt_lineages[p_gt] = [get_descendants(d, gt_adj, depth=3) for d in daughters[:2]]

    tp = 0
    matched_gt = set()
    
    # 3. Match predicted divisions to GT divisions with temporal (+/- 1 frame) and spatial (< 7um) tolerances
    for p_pred, c_pred in pred_divs.items():
        if p_pred not in pred_nodes_lookup.index:
            continue
        row_pred = pred_nodes_lookup.loc[p_pred]
        t_pred = int(row_pred['t'])
        pos_pred = np.array([row_pred['z'], row_pred['y'], row_pred['x']]) * SCALE
        
        best_gt = None
        best_dist = 1e6
        
        for p_gt, daughters in gt_divs.items():
            if p_gt in matched_gt:
                continue
            row_gt = gt_nodes_lookup.loc[p_gt]
            t_gt = int(row_gt['t'])
            
            # Temporal gating
            if abs(t_pred - t_gt) > 1:
                continue
                
            # Spatial gating
            pos_gt = np.array([row_gt['z'], row_gt['y'], row_gt['x']]) * SCALE
            dist = np.linalg.norm(pos_pred - pos_gt)
            if dist > r:
                continue
                
            # Lineage verification
            mapped_c = {p2g.get(c) for c in c_pred if c in p2g}
            lin1, lin2 = gt_lineages[p_gt]
            
            touches_lin1 = any(mc in lin1 for mc in mapped_c)
            touches_lin2 = any(mc in lin2 for mc in mapped_c)
            
            if touches_lin1 and touches_lin2:
                if dist < best_dist:
                    best_dist = dist
                    best_gt = p_gt
                    
        if best_gt is not None:
            tp += 1
            matched_gt.add(best_gt)
            
    n_gt = len(gt_divs)
    n_pred = len(pred_divs)
    fp = n_pred - tp
    fn = n_gt - tp
    
    return tp, fp, fn

def compute_metric(pred_df, gt_nodes, gt_edges, geff_path=None, r=7.0):
    pred_nodes = pred_df[pred_df.row_type=='node'][['node_id','t','z','y','x']].copy()
    pred_edges = pred_df[pred_df.row_type=='edge'][['source_id','target_id']].astype(int).copy()
    p2g  = _match_nodes(pred_nodes, gt_nodes, r)
    n_tp = len(p2g)
    n_fn = len(gt_nodes) - n_tp
    n_fp = len(pred_nodes) - n_tp
    
    gt_edge_set = set(map(tuple, gt_edges[['source_id','target_id']].values))
    pred_mapped = set()
    for _, row in pred_edges.iterrows():
        s, t_ = int(row['source_id']), int(row['target_id'])
        if s in p2g and t_ in p2g:
            pred_mapped.add((p2g[s], p2g[t_]))
    e_tp  = len(pred_mapped & gt_edge_set)
    e_fp  = len(pred_mapped - gt_edge_set)
    e_fn  = len(gt_edge_set - pred_mapped)
    e_jac = e_tp / max(e_tp+e_fp+e_fn, 1)
    
    T_pred  = len(pred_nodes)
    T_true  = read_geff_estimated_nodes(geff_path) if geff_path else (n_tp + n_fn)
    if T_true is None: T_true = n_tp + n_fn
    
    if T_pred > int(T_true):
        penalty = int(T_true) / T_pred
    else:
        penalty = 1.0
    adj_jac = e_jac * penalty
    
    # Compute division Jaccard metrics
    div_tp, div_fp, div_fn = compute_division_metrics(pred_df, gt_nodes, gt_edges, p2g, r)
    
    weight  = e_tp + e_fp + e_fn
    return adj_jac, {
        'node_recall':  round(n_tp / max(n_tp + n_fn, 1), 4),
        'node_prec':    round(n_tp / max(n_tp + n_fp, 1), 4),
        'edge_jac':     round(e_jac, 4),
        'adj_edge_jac': round(adj_jac, 4),
        'div_tp': div_tp, 'div_fp': div_fp, 'div_fn': div_fn,
        'T_pred': T_pred, 'T_true_est': int(T_true),
        'weight': weight,
    }

def run_validator(train_dir):
    if not RUN_VALIDATOR:
        print("Validator disabled"); return None
    all_names = list_datasets(train_dir)
    # Balanced validation subset
    names_44b6 = [n for n in all_names if n.startswith('44b6')][:2]
    names_6bba = [n for n in all_names if n.startswith('6bba')][:2]
    names = names_44b6 + names_6bba
    if len(names) == 0:
        names = all_names[:VALIDATOR_N]
        
    records = []
    sep = '─' * 72
    print(f"\n{sep}")
    print(f"  LOCAL VALIDATOR  |  {len(names)} samples (Dengeli Mix)  |  "
          f"thr={REL_THRESHOLD}  min_dist={MIN_DIST_UM}µm  "
          f"tight={TIGHT_UM}µm  loose={MAX_LINK_UM}µm  "
          f"min_track={MIN_TRACK_LEN}  smooth={DO_LINEFIT_SMOOTH}  "
          f"gap2={DO_RECOVER_GAP2}  safe_div={DO_SAFE_DIV}")
    print(sep)
    for nm in names:
        geff_path = os.path.join(train_dir, nm+'.geff')
        zarr_path = os.path.join(train_dir, nm+'.zarr')
        if not os.path.isdir(geff_path) or not os.path.isdir(zarr_path):
            print(f"  {nm}: missing — skipping"); continue
        gt_nodes, gt_edges = read_geff(geff_path)
        if gt_nodes is None:
            print(f"  {nm}: geff read failed — skipping"); continue
        shape, dtype = read_meta(zarr_path)
        t0v = time.time()
        nr, er, st  = track_movie(zarr_path, shape, dtype, nm)
        pred_df     = pd.DataFrame(nr+er)
        sc, det     = compute_metric(pred_df, gt_nodes, gt_edges, geff_path)
        print(f"\n  {nm}  ({time.time()-t0v:.0f}s)")
        print(f"    node_recall={det['node_recall']:.3f}  edge_jac={det['edge_jac']:.4f}"
              f"  adj_jac={det['adj_edge_jac']:.4f}")
        print(f"    T_pred={det['T_pred']:,}  T_true≈{det['T_true_est']:,}"
              f"  gap={st['n_gap']}  gap2={st['n_gap2']}  div={st['n_div']}"
              f"  filtered={st['n_filter']}")
        records.append({'dataset': nm, **det})
    if records:
        df = pd.DataFrame(records)
        w  = df['weight'].values.astype(np.float64)
        
        # 1. Weighted Edge Jaccard
        wtd_adj  = float((df['adj_edge_jac'].values * w).sum() / w.sum()) if w.sum() > 0 else 0.0
        
        # 2. Micro-averaged Division Jaccard
        tot_div_tp = df['div_tp'].sum()
        tot_div_fp = df['div_fp'].sum()
        tot_div_fn = df['div_fn'].sum()
        micro_div_jac = tot_div_tp / max(tot_div_tp + tot_div_fp + tot_div_fn, 1)
        
        # 3. Combined Score (Resmi LB Tahmini)
        combined_score = wtd_adj + 0.1 * micro_div_jac
        
        print(f"\n  ┌─ SUMMARY ───────────────────────────────────────────────┐")
        print(f"  │  Weighted Edge Jaccard (A)        : {wtd_adj:.4f}        │")
        print(f"  │  Micro Division Jaccard (B)       : {micro_div_jac:.4f}        │")
        print(f"  │  Combined Score (A + 0.1*B)       : {combined_score:.4f}        │")
        print(f"  │  Use Combined Score for LB estimate.                      │")
        print(f"  └──────────────────────────────────────────────────────────┘")
        print(df[['dataset', 'node_recall', 'edge_jac', 'adj_edge_jac', 'weight']]
              .to_string(index=False))
    print(f"\n{sep}\n")
    return records

print("Validator OK ✓")


# ── CELL 12: Local Validation ─────────────────────────────────────────────────

try:
    val_records = run_validator(TRAIN_DIR)
except Exception as e:
    print(f"Validator failed ({type(e).__name__}: {str(e)[:80]})")
    val_records = None


# ── CELL 13: Process All Test Datasets ───────────────────────────────────────

test_datasets = list_datasets(TEST_DIR)
print(f"\n{len(test_datasets)} test datasets found:")
for nm in test_datasets:
    print(f"  {nm}")

all_node_rows, all_edge_rows = [], []
t0 = time.time()

for i, nm in enumerate(test_datasets, 1):
    zarr_path    = os.path.join(TEST_DIR, nm+'.zarr')
    shape, dtype = read_meta(zarr_path)
    t1 = time.time()
    nr, er, st = track_movie(zarr_path, shape, dtype, nm)
    all_node_rows.extend(nr)
    all_edge_rows.extend(er)
    print(f"[{i}/{len(test_datasets)}] {nm}")
    print(f"    detect={st['n_detect']:,}  gap={st['n_gap']}  gap2={st['n_gap2']}"
          f"  div={st['n_div']}  filtered={st['n_filter']}  →  nodes={st['n_nodes']:,}  "
          f"edges={st['n_edges']:,}  {time.time()-t1:.0f}s  "
          f"(total {time.time()-t0:.0f}s)")

print(f"\nTotal: {time.time()-t0:.0f}s")


# ── CELL 14: Submission + Sanity Checks ──────────────────────────────────────

submission = pd.DataFrame(all_node_rows + all_edge_rows)
submission.index.name = 'id'
submission.to_csv('submission.csv')

n_nodes_total = (submission.row_type == 'node').sum()
n_edges_total = (submission.row_type == 'edge').sum()
print(f"\nsubmission.csv: {len(submission):,} rows"
      f"  (nodes: {n_nodes_total:,}  |  edges: {n_edges_total:,})")
print(f"Config: thr={REL_THRESHOLD}  min_dist={MIN_DIST_UM}µm"
      f"  tight={TIGHT_UM}µm  loose={MAX_LINK_UM}µm"
      f"  gap={MAX_GAP}/dist={GAP_DIST_UM}µm"
      f"  smooth_w={SMOOTH_W}  gap2={DO_RECOVER_GAP2}"
      f"  min_track={MIN_TRACK_LEN}  safe_div={DO_SAFE_DIV}"
      f"  bgsub_refine={USE_BGSUB_REFINE}  round_coords={ROUND_COORDS}")

print("\nSanity checks...")
assert list(submission.columns) == [
    'dataset','row_type','node_id','t','z','y','x','source_id','target_id']
assert set(test_datasets).issubset(set(submission.dataset.unique()))
assert (submission[submission.row_type=='node'][['t']] >= 0).all().all()
for ds, grp in submission.groupby('dataset'):
    valid = set(grp.loc[grp.row_type=='node','node_id'])
    e = grp[grp.row_type=='edge']
    assert e['source_id'].isin(valid).all(), f"Dangling source_id in {ds}"
    assert e['target_id'].isin(valid).all(), f"Dangling target_id in {ds}"
print("All sanity checks passed ✅")

print("\n── Dataset summary ────────────────────────────────────────────────")
for ds in test_datasets:
    grp = submission[submission.dataset==ds]
    n   = (grp.row_type=='node').sum()
    e   = (grp.row_type=='edge').sum()
    print(f"  {ds}:  {n:>7,} nodes  |  {e:>7,} edges")

print("\n🚀 Ready to submit!")